In [1]:
######################################################
### SIMULATION
######################################################
import time, importlib
import importlib
import python_modules.UeRanSim
import python_modules.Open5GS
import python_modules.Monitor
import python_modules.IperfApp
import python_modules.Results
importlib.reload( python_modules.UeRanSim )
importlib.reload( python_modules.Open5GS )
importlib.reload( python_modules.Monitor )
importlib.reload( python_modules.IperfApp )
importlib.reload( python_modules.Results )
from  python_modules.UeRanSim import UeRanSim
from  python_modules.Open5GS  import Open5GS
from  python_modules.Monitor  import Monitor
from  python_modules.IperfApp import IperfApp
from  python_modules.Results  import Results

host="mec"
uersim = UeRanSim(  ["001010000001001"] , "ue" )
mon    = Monitor()
iperf  = IperfApp( mon , uersim , apn_mec="10.2.0.1" , apn_cld="10.1.0.1" )
o5gs   = Open5GS( "172.17.0.2" ,"27017")

# Init subscribers
o5gs.removeAllSubscribers()
p = o5gs.getProfile_OneUe_maxThr( 100 )
p["imsi"] = "001010000001001"
o5gs.addSubscriber(p)

# Init UEs and required services; wait for initialization done
uersim.stop_ue( 0 )
uersim.start_ue( 0 , apn="all", sst="1", sd="1")
# mon.start_cpu_load_gen_process( "srv_mec" )
uersim.wait_ue_connection_up( 0 , host)
time.sleep(2)


*** Stop iperf servers
*** Stop iperf clients
*** Open5GS: Removing all subscribers
[13:27:17.239281] - Stop UE 0:
[13:27:17.367105] - Start UE 0 | imsi:imsi-001010000001001 | apn:all


In [2]:
t_start = time.time()
iperf.clean_all()
mon.clean_redis()
mon.set_start_sim(t_start)

mon.start_all_monitors()

time.sleep(1)

# iperf.start_session( ue_id=0, apn_name="mec" , sst="1" , prot="udp" , port="51000" , ul_dl="ul" , bwt="10" , t_dur="5" )
# time.sleep(3)
# iperf.start_session( ue_id=0, apn_name="cld" , sst="1" , prot="udp" , port="51001" , ul_dl="ul" , bwt="20" , t_dur="5" )

iperf.start_session( ue_id=0, apn_name="mec" , sst="1" , prot="tcp" , port="51000" , ul_dl="ul" , bwt=10 , t_dur=5 )
# while iperf.check_running_iperfs():
#     time.sleep(1)
time.sleep(5.5)
iperf.start_session( ue_id=0, apn_name="mec" , sst="1" , prot="tcp" , port="51000" , ul_dl="ul" , bwt=30 , t_dur=3 )

time.sleep(5)

import json
res = Results( f'results/iperf_test.json' )
res.collect_and_save_results( mon.redis_cli, mon, iperf )



*** Stop iperf servers
*** Stop iperf clients
Starting iperf:0 0 10.2.0.2 10.2.0.1 51000 tcp ul 10 5
Starting iperf:0 1 10.2.0.2 10.2.0.1 51000 tcp ul 30 3
*** Save results ... done.
